# Entrenamiento de la Red Siamesa

Este notebook ejecuta el script de entrenamiento y analiza los resultados.

**Notebooks previos requeridos:**
- `01_dataset_preparation` — prepara los frames y recorta las caras válidas.
- `02_pair_generation_preview` — genera los pares siameses de entrenamiento, validación y test.
- `03_siamese_model_summary` — describe la arquitectura del modelo.

Este notebook **no define** la arquitectura del modelo ni la lógica de entrenamiento.
Delega esa responsabilidad en `src/training/train.py`.

## Parámetros

In [ ]:
# --- Parámetros editables ---

# False por defecto para evitar lanzar un entrenamiento largo accidentalmente.
# Cambia a True cuando quieras ejecutar el entrenamiento.
RUN_TRAINING = False

EPOCHS        = 10
BATCH_SIZE    = 32
LEARNING_RATE = 0.0001
MODEL_NAME    = "siamese_model.keras"

# Si True, muestra el tamaño de los archivos generados al final
SHOW_MODEL_FILES = True

## Importaciones y configuración

In [ ]:
import sys
import subprocess
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Detectar la raíz del proyecto sin importar desde qué directorio se abre el notebook
_cwd = Path.cwd()
if (_cwd / 'src').exists():
    project_root = _cwd
elif (_cwd.parent / 'src').exists():
    project_root = _cwd.parent
else:
    raise RuntimeError(
        f'No se encontró el directorio src/. '
        f'Directorio actual: {_cwd}'
    )

sys.path.insert(0, str(project_root))

from src.config import (
    PROJECT_ROOT,
    PAIRS_DIR,
    LOGS_DIR,
    METRICS_DIR,
    SAVED_MODEL_DIR,
)

print(f'Python         : {sys.executable}')
print(f'project_root   : {project_root}')
print(f'PAIRS_DIR      : {PAIRS_DIR}')
print(f'LOGS_DIR       : {LOGS_DIR}')
print(f'METRICS_DIR    : {METRICS_DIR}')
print(f'SAVED_MODEL_DIR: {SAVED_MODEL_DIR}')

## Función auxiliar

In [ ]:
def run_command(command: list[str]) -> None:
    """Ejecuta un comando de shell e imprime stdout y stderr."""
    print('Comando:', ' '.join(command))
    print('-' * 60)
    result = subprocess.run(
        command,
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('[stderr]')
        print(result.stderr)
    print('-' * 60)
    print(f'Código de retorno: {result.returncode}')

## Verificación de prerequisitos

Los archivos CSV de pares deben existir antes de iniciar el entrenamiento.
Si alguno falta, ejecuta primero el notebook `02_pair_generation_preview` o el comando indicado.

In [ ]:
train_csv = PAIRS_DIR / 'train_pairs.csv'
val_csv   = PAIRS_DIR / 'val_pairs.csv'
test_csv  = PAIRS_DIR / 'test_pairs.csv'

missing = [p for p in [train_csv, val_csv, test_csv] if not p.exists()]

if missing:
    print('Archivos faltantes:')
    for p in missing:
        print(f'  {p}')
    print()
    print('Primero ejecuta el notebook 02 o el comando:')
    print('    python -m src.dataset.build_pairs --overwrite')
else:
    print('Todos los archivos de pares están presentes.')
    print(f'  {train_csv}')
    print(f'  {val_csv}')
    print(f'  {test_csv}')

## Resumen de pares

Distribución de pares positivos y negativos por split.

In [ ]:
if not missing:
    splits = {
        'train': pd.read_csv(train_csv),
        'val':   pd.read_csv(val_csv),
        'test':  pd.read_csv(test_csv),
    }

    summary_rows = []
    for split_name, df in splits.items():
        total    = len(df)
        positive = int((df['label'] == 1).sum())
        negative = int((df['label'] == 0).sum())
        summary_rows.append({
            'Split':    split_name,
            'Total':    total,
            'Positivos': positive,
            'Negativos': negative,
        })

    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
else:
    print('No se pueden cargar los CSVs porque faltan archivos.')

## Comando de entrenamiento

Si `RUN_TRAINING = True`, lanza el script `src/training/train.py` con los parámetros definidos arriba.

Si `RUN_TRAINING = False`, imprime el comando que se ejecutaría, sin lanzarlo.

In [ ]:
training_command = [
    sys.executable, '-m', 'src.training.train',
    '--epochs',        str(EPOCHS),
    '--batch-size',    str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--model-name',    MODEL_NAME,
]

if RUN_TRAINING:
    run_command(training_command)
else:
    print('RUN_TRAINING = False — entrenamiento no ejecutado.')
    print()
    print('Comando que se ejecutaría:')
    print(' ', ' '.join(training_command))

## Artefactos generados

Rutas esperadas después del entrenamiento y estado actual en disco.

In [ ]:
model_stem   = Path(MODEL_NAME).stem
best_model   = SAVED_MODEL_DIR / MODEL_NAME
final_model  = SAVED_MODEL_DIR / f'final_{MODEL_NAME}'
training_log = LOGS_DIR        / f'{model_stem}_training.csv'
history_json = METRICS_DIR     / f'{model_stem}_history.json'

artifacts = [
    ('Mejor modelo (checkpoint)', best_model),
    ('Modelo final',              final_model),
    ('Log CSV (CSVLogger)',       training_log),
    ('Historial JSON',            history_json),
]

status_rows = [
    {'Artefacto': label, 'Ruta': str(path), 'Existe': path.exists()}
    for label, path in artifacts
]
display(pd.DataFrame(status_rows))

## Historial de entrenamiento

Carga el JSON generado por `save_history()` al finalizar el entrenamiento.

In [ ]:
history_df = None

if history_json.exists():
    with open(history_json, 'r', encoding='utf-8') as f:
        history_data = json.load(f)

    print('Métricas disponibles:', list(history_data.keys()))
    history_df = pd.DataFrame(history_data)
    history_df.index.name = 'epoch'
    display(history_df)
else:
    print('El archivo history JSON no existe todavía.')
    print('Ejecuta el entrenamiento con RUN_TRAINING = True para generarlo.')

## Curvas de entrenamiento

In [ ]:
if history_df is not None:
    epochs_range = range(1, len(history_df) + 1)

    # --- Pérdida ---
    fig, ax = plt.subplots()
    ax.plot(epochs_range, history_df['loss'],     label='train')
    ax.plot(epochs_range, history_df['val_loss'], label='val')
    ax.set_title('Pérdida por época')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # --- Exactitud binaria ---
    if 'binary_accuracy' in history_df.columns:
        fig, ax = plt.subplots()
        ax.plot(epochs_range, history_df['binary_accuracy'],     label='train')
        ax.plot(epochs_range, history_df['val_binary_accuracy'], label='val')
        ax.set_title('Exactitud binaria por época')
        ax.set_xlabel('Época')
        ax.set_ylabel('Binary Accuracy')
        ax.legend()
        plt.tight_layout()
        plt.show()

    # --- Precisión ---
    if 'precision' in history_df.columns:
        fig, ax = plt.subplots()
        ax.plot(epochs_range, history_df['precision'],     label='train')
        ax.plot(epochs_range, history_df['val_precision'], label='val')
        ax.set_title('Precisión por época')
        ax.set_xlabel('Época')
        ax.set_ylabel('Precision')
        ax.legend()
        plt.tight_layout()
        plt.show()

    # --- Recall ---
    if 'recall' in history_df.columns:
        fig, ax = plt.subplots()
        ax.plot(epochs_range, history_df['recall'],     label='train')
        ax.plot(epochs_range, history_df['val_recall'], label='val')
        ax.set_title('Recall por época')
        ax.set_xlabel('Época')
        ax.set_ylabel('Recall')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print('No hay historial disponible para graficar.')

## Log CSV del entrenamiento

El callback `CSVLogger` registra cada época en tiempo real.
Útil para revisar el progreso sin cargar el JSON completo.

In [ ]:
if training_log.exists():
    log_df = pd.read_csv(training_log)

    print(f'Total de épocas registradas: {len(log_df)}')
    print()
    print('Últimas épocas:')
    display(log_df.tail(10))

    if 'val_loss' in log_df.columns:
        best_epoch = log_df['val_loss'].idxmin()
        print()
        print(f'Mejor época (val_loss mínimo): época {best_epoch + 1}')
        display(log_df.iloc[[best_epoch]])
else:
    print('El log CSV no existe todavía.')
    print('Ejecuta el entrenamiento con RUN_TRAINING = True para generarlo.')

## Archivos del modelo guardado

In [ ]:
if SHOW_MODEL_FILES:
    model_files = list(SAVED_MODEL_DIR.glob('*')) if SAVED_MODEL_DIR.exists() else []

    if model_files:
        file_rows = []
        for p in sorted(model_files):
            size_mb = p.stat().st_size / (1024 ** 2) if p.is_file() else None
            file_rows.append({
                'Archivo': p.name,
                'Tamaño (MB)': f'{size_mb:.2f}' if size_mb is not None else '—',
            })
        display(pd.DataFrame(file_rows))
    else:
        print('No hay archivos de modelo en:', SAVED_MODEL_DIR)
        print('Ejecuta el entrenamiento con RUN_TRAINING = True.')

## Interpretación de las curvas

- **Pérdida decreciente en train** indica que el modelo está aprendiendo a distinguir pares.
- **val_loss** es la señal usada por `EarlyStopping` para detener el entrenamiento.
  El mejor checkpoint corresponde a la época con `val_loss` mínimo.
- Si **train_loss baja pero val_loss sube**, hay sobreajuste: el modelo memoriza los pares
  de entrenamiento sin generalizar.
  Acciones posibles: reducir épocas, aumentar dropout, o generar más pares negativos.
- Las métricas finales (precisión, recall, AUC, umbral óptimo) se calculan en el
  notebook de evaluación con el split de test.

## Lista de verificación

- [ ] Pares generados (`train_pairs.csv`, `val_pairs.csv`, `test_pairs.csv`).
- [ ] Entrenamiento ejecutado (`RUN_TRAINING = True`).
- [ ] Mejor modelo guardado en `models/saved_model/<model_name>`.
- [ ] Modelo final guardado en `models/saved_model/final_<model_name>`.
- [ ] Curvas de entrenamiento revisadas.
- [ ] **Outputs limpios antes del commit** (`Kernel → Restart & Clear Output`).
- [ ] Siguiente paso: notebook de evaluación (`05_evaluation`).